# 10 — Recursive Depth

When a team produces a high-novelty finding, the engine spawns a child team at `depth+1`. The child runs with a different prompt that shifts from "map the territory" to "drill the mechanism" to "empirical specifics."

Three mechanisms keep the tree focused:
- **Novelty threshold** — only findings above `novelty_threshold` (default 0.7) trigger children.
- **Topic deduplication** — `_seen_topics` prevents re-investigating the same question.
- **Depth budget** — `max_depth` (default 3) caps the tree.

In [ ]:
from lionag2.research.prompts import build_node_instruction

## How children spawn

Bridge observers on each agent drive spawning:

```
Agent calls emit_finding(claim=..., novelty=0.85)
  → ctx.send(FindingEmitted)
  → @agent.observer(FindingEmitted) fires
  → novelty > threshold → engine._spawn_depth_node()
  → check: new_depth <= max_depth? topic not seen?
  → child team runs as asyncio.Task
```

Multiple children can run concurrently (up to `max_concurrent`).

## Topic deduplication

```python
normalized = topic.strip().lower()
if self._topic_seen(normalized):
    return
```

The full topic string (lowercased, stripped) serves as a deduplication key. This prevents parallel branches from spawning identical investigations.

## Depth-aware prompts

Each depth level has a different goal:

In [ ]:
topic = "Does the magnetic resonance energy track Tc?"

for depth in range(4):
    instruction = build_node_instruction(topic, depth=depth, max_depth=3)
    # Extract just the depth guidance section
    lines = instruction.split("\n")
    for line in lines:
        if "Depth guidance" in line or "depth=" in line.lower():
            print(f"depth={depth}: {line.strip()}")
            break

## Context carry-over

Child teams receive parent findings as context, filtered by `node_id` to scope to the specific parent branch:

```python
parent_findings = [
    f for f in flow.items.by_type(FindingEmitted)
    if f.node_id == parent_node_id
]
context = "\n".join(f"- [{f.source_agent}] {f.claim}" for f in parent_findings)
```

Deeper nodes see only their parent's discoveries, preventing context dilution while preserving key insights.

## Waiting for the tree

After the root team finishes, the engine waits for all reactive children to settle:

```python
while self._active_tasks:
    await asyncio.gather(*self._active_tasks, return_exceptions=True)
```

Only then does it proceed to cross-check and paper writing. Children can spawn grandchildren — the loop keeps waiting until the entire tree quiesces.

## Up next

Tutorial 11 shows how to serve lionag2 over the ag-ui protocol — connecting it to CopilotKit, Vercel AI SDK, or any ag-ui frontend.